In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import numpy as np
from openai import OpenAI
import shutil
import jieba
import os
import re

C:\Users\ASUS\miniconda3\envs\PythonProject\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (None) doesn't match a supported version!
  warnings.warn(
C:\Users\ASUS\miniconda3\envs\PythonProject\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ASUS\miniconda3\envs\PythonProject\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
def simple_clean(text):
    if not isinstance(text, str):  # 增加空值/非字符串防护
        return ""

    # 1. 基础格式清洗
    text = re.sub(r"\n+", " ", text)  # 删除换行
    text = re.sub(r"\s+", " ", text)  # 删除多余空格

    # 2. 修复断词（中文+英文）
    text = re.sub(r"-\s+", "", text)  # 修复中文断词（如“强- 化学习”→“强化学习”）
    text = re.sub(r'([a-zA-Z])-([a-zA-Z])', r'\1\2', text)  # 修复英文断词（如informa-tion→information）

    # 3. 剔除无意义标识
    text = re.sub(r"\[\d+\]", "", text)  # 删除参考文献编号（如[1]、[23]）
    text = re.sub(r"Page \d+", "", text)  # 删除页码（Page 123）
    # text = re.sub(r'\d{4,}', '', text)  # 删除长度≥4的纯数字串
    text = re.sub(r'[a-zA-Z0-9]{12,}', '', text)  # 删除字母数字混合串（如ISBN9787111）

    # 4. 剔除装饰性符号
    text = re.sub(r'[★■●※▲△■◆◇€£¥¢§#@&*()〔〕〖〗〘〙〚〛]', '', text)

    # 5. 剔除冗余标签
    redundant_tags = ['版权所有', '翻印必究']
    for tag in redundant_tags:
        text = text.replace(tag, '')

    # 6. 文本优化
    text = re.sub(r'(?<=[\u4e00-\u9fa5])\s+(?=[\u4e00-\u9fa5])', '', text)  # 删除中文之间的空格
    text = re.sub(r'([，。！？；：""''、,.!?;:]){2,}', r'\1', text)  # 剔除连续标点
    text = re.sub(r'([\u4e00-\u9fa5]{2,})\1', r'\1', text)  # 删除重复词（如“强化学习强化学习”→“强化学习”）
    text = re.sub(r'([\u4e00-\u9fa5])\1+', r'\1', text)  # 删除重复字（如“强强化化”→“强化”）

    # 7. 删除 1.问题？ 2.问题？
    text = re.sub(r'[（(]?\d+[）)]?\s*[^。！？?]*[？?]', '', text)
    text = re.sub(r'[（(]?\d+[）)\.]?\s*(?:[^。！？?]*[？?]\s*)+', '', text)

    # 8. 最终过滤
    cleaned_text = text.strip()
    return cleaned_text if len(cleaned_text) >= 3 else ""

def hybrid_retrieve(query, k=3, bm25_weight=0.5, vector_weight=0.5):
    # BM25
    chz_cut_bm25_retriever.k = k
    bm25_docs = chz_cut_bm25_retriever.invoke(query)

    # Vector
    vector_retriever = vector_db.as_retriever(search_kwargs={"k": k})
    vector_docs = vector_retriever.invoke(query)

    scores = {}
    doc_map = {}

    # BM25打分
    for rank, doc in enumerate(bm25_docs, 1):
        key = (doc.page_content, tuple(sorted(doc.metadata.items())))
        score = bm25_weight * (1 / (rank + 60))  # RRF核心
        scores[key] = scores.get(key, 0) + score
        doc_map[key] = doc

    # Vector打分
    for rank, doc in enumerate(vector_docs, 1):
        key = (doc.page_content, tuple(sorted(doc.metadata.items())))
        score = vector_weight * (1 / (rank + 60))
        scores[key] = scores.get(key, 0) + score
        doc_map[key] = doc

    # 排序
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [doc_map[key] for key, _ in ranked[:k]]

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def filter_chunk_by_query(chunk_text, query, embedding_model, threshold=0.5):
    # 1. 拆句
    sentences = re.split(r'[。！？]', chunk_text)

    # 2. query向量
    query_vec = embedding_model.embed_query(query)

    selected = []

    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 5:
            continue

        sent_vec = embedding_model.embed_query(sent)

        sim = cosine_sim(query_vec, sent_vec)

        if sim > threshold:
            selected.append(sent)

    # 3. 拼接
    return "。".join(selected)

In [3]:
def simple_clean(text):
    if not isinstance(text, str):  # 增加空值/非字符串防护
        return ""

    # 1. 基础格式清洗
    text = re.sub(r"\n+", " ", text)  # 删除换行
    text = re.sub(r"\s+", " ", text)  # 删除多余空格

    # 2. 修复断词（中文+英文）
    text = re.sub(r"-\s+", "", text)  # 修复中文断词（如“强- 化学习”→“强化学习”）
    text = re.sub(r'([a-zA-Z])-([a-zA-Z])', r'\1\2', text)  # 修复英文断词（如informa-tion→information）

    # 3. 剔除无意义标识
    text = re.sub(r"\[\d+\]", "", text)  # 删除参考文献编号（如[1]、[23]）
    text = re.sub(r"Page \d+", "", text)  # 删除页码（Page 123）
    # text = re.sub(r'\d{4,}', '', text)  # 删除长度≥4的纯数字串
    text = re.sub(r'[a-zA-Z0-9]{12,}', '', text)  # 删除字母数字混合串（如ISBN9787111）

    # 4. 剔除装饰性符号
    text = re.sub(r'[★■●※▲△■◆◇€£¥¢§#@&*()〔〕〖〗〘〙〚〛]', '', text)

    # 5. 剔除冗余标签
    redundant_tags = ['版权所有', '翻印必究']
    for tag in redundant_tags:
        text = text.replace(tag, '')

    # 6. 文本优化
    text = re.sub(r'(?<=[\u4e00-\u9fa5])\s+(?=[\u4e00-\u9fa5])', '', text)  # 删除中文之间的空格
    text = re.sub(r'([，。！？；：""''、,.!?;:]){2,}', r'\1', text)  # 剔除连续标点
    text = re.sub(r'([\u4e00-\u9fa5]{2,})\1', r'\1', text)  # 删除重复词（如“强化学习强化学习”→“强化学习”）
    text = re.sub(r'([\u4e00-\u9fa5])\1+', r'\1', text)  # 删除重复字（如“强强化化”→“强化”）

    # 7. 删除 1.问题？ 2.问题？
    text = re.sub(r'[（(]?\d+[）)]?\s*[^。！？?]*[？?]', '', text)
    text = re.sub(r'[（(]?\d+[）)\.]?\s*(?:[^。！？?]*[？?]\s*)+', '', text)

    # 8. 最终过滤
    cleaned_text = text.strip()
    return cleaned_text if len(cleaned_text) >= 3 else ""

def hybrid_retrieve(query, k=3, bm25_weight=0.5, vector_weight=0.5):
    # BM25
    chz_cut_bm25_retriever.k = k
    bm25_docs = chz_cut_bm25_retriever.invoke(query)

    # Vector
    vector_retriever = vector_db.as_retriever(search_kwargs={"k": k})
    vector_docs = vector_retriever.invoke(query)

    scores = {}
    doc_map = {}

    # BM25打分
    for rank, doc in enumerate(bm25_docs, 1):
        key = (doc.page_content, tuple(sorted(doc.metadata.items())))
        score = bm25_weight * (1 / (rank + 60))  # RRF核心
        scores[key] = scores.get(key, 0) + score
        doc_map[key] = doc

    # Vector打分
    for rank, doc in enumerate(vector_docs, 1):
        key = (doc.page_content, tuple(sorted(doc.metadata.items())))
        score = vector_weight * (1 / (rank + 60))
        scores[key] = scores.get(key, 0) + score
        doc_map[key] = doc

    # 排序
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [doc_map[key] for key, _ in ranked[:k]]

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def filter_chunk_by_query(chunk_text, query, embedding_model, threshold=0.5):
    # 1. 拆句
    sentences = re.split(r'[。！？]', chunk_text)

    # 2. query向量
    query_vec = embedding_model.embed_query(query)

    selected = []

    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 5:
            continue

        sent_vec = embedding_model.embed_query(sent)

        sim = cosine_sim(query_vec, sent_vec)

        if sim > threshold:
            selected.append(sent)

    # 3. 拼接
    return "。".join(selected)

In [4]:
# 1. 读取 PDF
loader = DirectoryLoader(
    r"D:\PycharmProjects\PythonProject\Question2\ss1", # 替换为指定PDF路径
    glob="*.pdf",
    loader_cls=PyPDFLoader
)

docs = loader.load()

# 清洗文本
for doc in docs:
    doc.page_content = simple_clean(doc.page_content)

# 2. 文本切块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)
split_docs = text_splitter.split_documents(docs)

# 3.1 构建 中文BM25 检索器
chz_cut_bm25_retriever = BM25Retriever.from_documents(
    split_docs,
    preprocess_func=lambda text: list(jieba.cut(text))
)
chz_cut_bm25_retriever.k = 3

# 3.2 构建embedding模型
embedding_model = HuggingFaceEmbeddings( # 初始化embedding模型
    model_name="BAAI/bge-small-zh"
)
persist_dir = "vector_db"
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)


vector_db = Chroma.from_documents(  # 构建向量数据库
    split_docs,
    embedding=embedding_model,
    persist_directory="vector_db"   # 持久化
)

vector_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 3.3 构建 Hybrid 检索器


# 4. 测试检索
def test_bm25(query):
    print(f"用户问题: {query}\n")
    print("BM25 召回结果:")

    results = chz_cut_bm25_retriever.invoke(query)

    for i, doc in enumerate(results, 1):
        print(f"{i}. {doc.page_content}")
        print(f"metadata: {doc.metadata}")
        print()

    print("-" * 50)

def test_vector(query, k=3):
    print(f"用户问题: {query}\n")
    print("Embedding 召回结果:")

    docs_and_scores = vector_db.similarity_search_with_relevance_scores(query, k=k)

    for i, (doc, score) in enumerate(docs_and_scores, 1):
        print(f"{i}. {doc.page_content}")
        print(f"metadata: {doc.metadata}")
        print(f"score: {score:.4f}")
        print()

def test_hybrid(query, k=3, bm25_weight=0.5, vector_weight=0.5):
    print(f"用户问题: {query}\n")
    print("Hybrid 召回结果:")

    results = hybrid_retrieve(query, k, bm25_weight, vector_weight)

    for i, doc in enumerate(results, 1):
        filtered = filter_chunk_by_query(
            doc.page_content,
            query,
            embedding_model,
            threshold=0.5
        )

        print(f"{i}. {filtered}")
        print()
    print("-" * 50)

def rag_answer(query, k=5):
    # 1. 混合检索
    results = hybrid_retrieve(query, k=k)

    # 2. 拼接检索到的上下文
    context = "\n\n".join([
        f"[片段{i+1}]\n{doc.page_content}"
        for i, doc in enumerate(results)
    ])

        # 3. 构造 messages
    messages = [
        {
            "role": "system",
            "content": (
                "你是一个海运行业的知识问答助手，请严格依据给定资料回答问题，不要凭空编造。"
                "请根据题目作答，每个题目都是单选题，请认真理解每个选项，根据资料和理性理解确定合适题目要求的答案，只能输出 [A] 或 [B] 或 [C] 或 [D] 之一，不能输出其他内容，并确保你的最终答案和推理过程保持一致。"
                "此时为测试阶段，输出正确答案后，并给出你的答题逻辑。"
            )
        },
        {
            "role": "user",
            "content": f"""已知资料：
        {context}

        用户问题：
        {query}
        """
        }
    ]
    client = OpenAI(
    # 如果没有配置环境变量，请用阿里云百炼API Key替换：api_key="sk-xxx"
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
    # 4. 调用大模型
    completion = client.chat.completions.create(
        model="qwen3-8b",
        messages=messages,
        extra_body={"enable_thinking": False}
    )

    answer = completion.choices[0].message.content

    return answer, results

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\ASUS\AppData\Local\Temp\jieba.cache
Loading model cost 0.501 seconds.
Prefix dict has been built successfully.
C:\Users\ASUS\AppData\Local\Temp\ipykernel_6728\1414703524.py:29: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings( # 初始化embedding模型

KeyboardInterrupt



In [ ]:
if __name__ == "__main__":

    # print("====== BM25 ======")
    # test_bm25("脉管系统是")
    # test_bm25("细胞")

    # print("====== Embedding ======")
    # test_vector("什么是脉管系统？")
    # test_vector("什么是细胞？")

    # print("====== HYBRID ======")
    # test_hybrid("什么是脉管系统?")
    # test_hybrid("什么是细胞?")

    ### 一共20个问题，供测试使用，答案都可在当前教材中找到
    questions = [
        "浅水船舶操纵特性的变化趋势是______。①稳定性变好、旋回性变差②下沉量增大③冲程增大④舵效变差。 [A]①③ [B]①②③④ [C]②③④ [D]①②④",
        "下列___是保证船舶局部强度不受损伤的措施。①装载重大件货物时应加适当衬垫②货物在舱内应均匀分布③按照舱容比分配各舱货物的重量④按船舶的腐蚀程度确定甲板允许负荷⑤限制重货落底速度。 [A]②③④ [B]①②④⑤ [C]①②③④⑤ [D]①③⑤",
        "板为平直甲板且货舱为单层甲板的船是____。 [A]木材船 [B]滚装船 [C]散装船 [D]集装箱船",
        "某船船长Lbp=140m，实测船舶首、尾吃水分别为8.54m、9.28m，船中两舷吃水分别为8.64m、9.28m，则船舶____。 [A]中垂，纵强度满足要求 [B]中拱，纵强度满足要求 [C]中垂，纵强度不满足要求 [D]中拱，纵强度不满足要求",
        "《中华人民共和国防治船舶污染海洋环境管理条例》规定，将使用完毕的含油污水、含有毒有害物质污水记录簿在船舶上保留______年。[A]1 [B]2 [C]3 [D]5",
        "根据船长的大小配备信号旗，船长大于100米的船舶应配备国际信号旗几面________。[A]10 [B]20 [C]30 [D]40",
        "充气式救生筏的充气钢瓶里气体是二氧化碳和少部分氮气，氮气的作用是______。[A]增加气体压力 [B]降低二氧化碳的毒性 [C]防止充气时管路结冰",
        "ISM规则是一个强制实施的______。[A]IMO通过的船舶结构技术标准 [B]安全与防污染管理国际标准 [C]国际安全与防污染管理体系 [D]IMO通过的船舶设备技术标准",
        "船舶保安审核的种类不包括______。[A]初次审核 [B]年度审核 [C]中间审核 [D]换证审核",
        "下列有关影响旋回圈大小因素的叙述哪项是错误的？ [A]方形系数大的船，旋回圈小 [B]有球鼻首的船，旋回圈较小 [C]船舶重载时，旋回初径有所减小 [D]浅水中旋回时，旋回圈变大",
        "______ are used in some installations where the cable lifter rotates about a vertical axis. [A]Windlasses [B]Drums [C]Anchor capstans [D]Warp ends",
        "一艘正常航行的机动船的航行灯包括______。 [A]桅灯、舷灯、尾灯 [B]桅灯、舷灯、号灯 [C]前、后桅灯，左、右舷灯，环照灯",
        "从事捕鱼的船舶”是指使用网具、绳的、拖网或其他______的渔具从事捕鱼的船舶。 [A]按照本规则条款的要求进行操纵的能力受到限制 [B]使其操纵性能受到限制，因而不能给他船让路 [C]使其操纵性能受到限制 [D]按照本规则条款的要求进行操纵的能力受到限制，因而不能给他船让路",
        "根据“责任”条款规定，关于船舶应当考虑的可能导致背离《规则》采取行动的危险和特殊情况，下列哪项说法是正确的？ [A]包括当事船舶的条件限制在内 [B]指同时存在特殊情况和紧迫危险 [C]仅限于紧迫危险 [D]不包括碰撞的危险",
        "气胀式救生筏的属具中，要求应供有足够______的救生筏额定乘员使用的保温用具。 [A]10% [B]5% [C]15%",
        "SOLAS公约规定，在弃船演习时每艘救生艇应______。 [A]每个月降落下水一次 [B]每3个月降落一次 [C]每3个月降落下水一次 [D]每6个月降落下水一次",
        "救生艇内的淡水（密封罐装的除外）应每隔多少时间更换一次？ [A]一个月 [B]六个月 [C]十二个月",
        "保持船级的检验中，特别检验的间隔期是______。 [A]1年 [B]2年 [C]3年 [D]5年",
        "救生筏的自行更换式静水压力释放器每______更换一次。 [A]1年 [B]2年 [C]5年"
    ]

    # 测试混合检索，召回top-20
    # for question in questions:
    #     test_hybrid(question, k=20)
    #
    #
    #     answer, docs = rag_answer(question)
    #     print("=== 回答 ===")
    #     print(answer)
    #     print(docs)


In [ ]:
answer, docs = rag_answer(questions[2])
    #     print("=== 回答 ===")
    #     print(answer)
    #     print(docs)